# Within-lane probe — is the wall the inputs, or the cross-lane structure?

The held-out-lane probe ([[encoder_router_os_distill_relabel_probe]]) measured a router
that must generalize to an unseen collection: every arm landed BELOW the shuffled floor.
Two rival explanations survive that result, and this probe separates them by removing
cross-lane contradictions **by construction**: split 80/20 at random WITHIN each lane,
train on the 80% of every lane, evaluate on the 20% of the same lanes. Same query
distribution, same labels, no hidden-variable flips between train and test.

**Decision rules (read §3 against these):**
- Router beats the per-lane constants comfortably within-lane → labels AND inputs are
  sufficient; the generic router's only obstacle is cross-lane label structure →
  train on the cross-lane-consistent subset (census next).
- Router can't beat per-lane constants even here → contradictions were never binding;
  the input representation (rarity blindness) is the wall → feature work before any
  further label spend.
- The `decisive_only` arm vs `no_branches`: whether the 24K qrels-ceiling tie rows
  (targets = "both acceptable") dilute the heads — the tie-hygiene ablation.

The per-lane constant is computed from each lane's TRAIN rows — within this protocol it
is a fair, non-leaking baseline (diagnosis, not deployability, is the question here).

In [2]:
from __future__ import annotations

import os
# torch, sklearn, and lightgbm each ship their own libomp on macOS; loading two
# in one process corrupts OpenMP barriers -> SIGSEGV in the next parallel op
# (seen: kernel death inside torch.ones during EncoderRouter.fit). Must be set
# BEFORE the first import of any of them.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

from pathlib import Path

import numpy as np
import pandas as pd

from hybrid_search_rrf_dataset.labels import AcceptabilityLabels, outcome_shape
from encoder_router.table import KEY, MODELS_DIR, ROUTES
from encoder_router.training import TrainingTable


def find_data_dir(start: Path) -> Path:
    for directory in (start.resolve(), *start.resolve().parents):
        candidate = directory / "src" / "data"
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(f"could not find src/data above {start}")


DATA_DIR = find_data_dir(Path.cwd())
CASCADE_LABELS = DATA_DIR / "legb_pilot" / "os_distill_relabel" / "cascade_labels.parquet"
SEED, TEST_FRAC = 0, 0.2

cascade_raw = pd.read_parquet(CASCADE_LABELS)
cascade_raw["query_id"] = cascade_raw["query_id"].astype(str)
has_text = cascade_raw["query"].notna() & cascade_raw["query"].astype(str).str.strip().ne("")
cascade_trainable = cascade_raw.loc[has_text].reset_index(drop=True)


def build_table(cascade_labels: pd.DataFrame) -> TrainingTable:
    merged = AcceptabilityLabels(cascade_labels, tolerance=None).frame()
    score_columns = [f"score_{route}" for route in ROUTES]
    merged["shape"] = [
        outcome_shape(dict(zip(ROUTES, scores)))
        for scores in merged[score_columns].to_numpy(dtype=float)
    ]
    table = TrainingTable()
    table.__dict__["frame"] = merged.reset_index(drop=True)
    return table


table = build_table(cascade_trainable)
frame = table.frame
print(f"training table: {len(frame):,} rows, {frame['dataset'].nunique()} lanes")
print(frame["shape"].value_counts().to_string())

training table: 91,080 rows, 46 lanes
shape
routes_differ    57456
all_tied         23941
all_zero          9683


In [4]:
# Within-lane 80/20 split — stratified per lane, fixed seed, so every lane
# contributes both train and test rows and reruns reproduce the same split.
rng = np.random.default_rng(SEED)
test_mask = np.zeros(len(frame), dtype=bool)
for _, idx in frame.groupby("dataset").indices.items():
    picked = rng.choice(idx, size=max(1, int(len(idx) * TEST_FRAC)), replace=False)
    test_mask[picked] = True
train_mask = ~test_mask

dec = (frame["shape"] == "routes_differ") & frame["serve"].notna()
print(f"train {train_mask.sum():,} / test {test_mask.sum():,} rows")
print(f"decisive test rows (the evaluation set): {(test_mask & dec).sum():,} "
      f"across {frame.loc[test_mask & dec, 'dataset'].nunique()} lanes")

train 72,882 / test 18,198 rows
decisive test rows (the evaluation set): 11,465 across 46 lanes


In [6]:
# Train one model per arm on the within-lane TRAIN rows; save under models/within_lane_probe/.
# Mirrors the held-out-lane probe's train_arm, with a row-level pool mask instead of a
# lane holdout — SVD, z-stats, thresholds, and branch targets all fit on the pool only.
import json as _json, time as _time
from pathlib import Path as _P

from encoder_router.evaluate import Arm, LaneCV, tuned_thresholds
from encoder_router.model import EncoderRouter
from encoder_router.table import (EMBEDDING_PREFIXES, HEAD_ROUTES, NgramSvd,
                                  ZipfStats, serve_from_probabilities)
from encoder_router.training import QueryEmbeddings

TRAIN_VERSION = "2026-09-08-within-lane-80-20-v1"
BASE = MODELS_DIR / "within_lane_probe"
FORCE_RETRAIN = False

# (arm, extra pool restriction). decisive_only drops the qrels-ceiling tie rows from
# the route loss — the tie-hygiene ablation riding along at zero extra design cost.
PROBE_ARMS = (
    (Arm("no_branches",      cell_branch=False, corpus_branch=False), None),
    (Arm("shuffled_targets", shuffle_targets=True, corpus_branch=False), None),
    (Arm("zipf_input",       zipf_inputs=True, cell_branch=False, corpus_branch=False), None),
    (Arm("decisive_only",    cell_branch=False, corpus_branch=False), "routes_differ"),
)


def _zstats(a):
    m = a.mean(0); s = a.std(0); s[s == 0] = 1.0
    return m.astype(np.float32), s.astype(np.float32)


def train_arm_wl(table, arm, pool_mask, base_dir, seed=SEED):
    t0 = _time.perf_counter()
    def log(msg): print(f"  [{arm.name}] +{_time.perf_counter()-t0:6.1f}s  {msg}", flush=True)
    frame = table.frame
    pool = np.flatnonzero(pool_mask)
    log(f"train on {len(pool):,}/{len(frame):,} rows (within-lane pool)")
    emb = QueryEmbeddings(arm.embedding_model).matrix(frame)
    svd = NgramSvd(seed=seed).fit(frame.loc[pool_mask, "query"])
    blocks, stats = [emb, svd.transform(frame["query"])], {}
    if arm.zipf_inputs:
        z = ZipfStats().frame(frame["query"]).to_numpy(np.float32)
        m, s = _zstats(z[pool_mask]); stats["zipf"] = (m, s); blocks.append((z - m) / s)
    x = np.concatenate(blocks, axis=1).astype(np.float32)
    route = table.route_targets().to_numpy(np.float32)
    cell_t, corpus_t, feat_t = LaneCV(table, seed=seed)._targets(arm, pool_mask)
    if arm.shuffle_targets:
        rp = np.random.default_rng(seed + 1).permutation(len(pool))
        route = route.copy(); route[pool] = route[pool][rp]
        log("shuffled ROUTE labels within pool (permutation floor)")
    rng = np.random.default_rng(seed); perm = rng.permutation(len(pool)); cut = max(len(pool) // 10, 1)
    fi, vi = pool[perm[cut:]], pool[perm[:cut]]
    er = EncoderRouter(seed=seed).fit(
        x[fi], route[fi],
        None if cell_t is None else cell_t[fi],
        None if corpus_t is None else corpus_t[fi],
        None if feat_t is None else feat_t[fi],
        x_val=x[vi], val_route_targets=route[vi])
    log(f"mlp done: {len(er.history)} epochs, best_val={getattr(er, 'best_val_loss', float('nan')):.4f}")
    probs_tr = er.probabilities(x[pool])
    thr = tuned_thresholds(probs_tr, frame.loc[pool_mask])
    p = _P(base_dir) / arm.name; p.mkdir(parents=True, exist_ok=True)
    er.save(p / "router.pt"); svd.save(p / "svd.joblib"); np.save(p / "thresholds.npy", thr)
    for k, (m, s) in stats.items():
        np.save(p / f"{k}_mean.npy", m); np.save(p / f"{k}_std.npy", s)
    (p / "meta.json").write_text(_json.dumps(
        {"arm": arm.name, "learner": arm.learner, "embedding_model": arm.embedding_model,
         "prefix": EMBEDDING_PREFIXES.get(arm.embedding_model, ""),
         "zipf_inputs": arm.zipf_inputs, "split": "within-lane-80-20", "seed": seed,
         "trained_at": _time.strftime("%Y-%m-%d %H:%M:%S"), "train_version": TRAIN_VERSION}))
    log(f"SAVED -> {p}  ({_time.perf_counter()-t0:.1f}s)")


def _cache_ok(arm, base):
    p = base / arm.name; mp = p / "meta.json"
    if not mp.exists() or not (p / "router.pt").exists():
        return False
    try:
        m = _json.loads(mp.read_text())
    except Exception:
        return False
    return m.get("train_version") == TRAIN_VERSION and m.get("arm") == arm.name


for arm, restrict in PROBE_ARMS:
    if not FORCE_RETRAIN and _cache_ok(arm, BASE):
        print(f"[{arm.name}] cached, skip")
        continue
    pool = train_mask.copy()
    if restrict == "routes_differ":
        pool &= ((frame["shape"] == "routes_differ") & frame["serve"].notna()).to_numpy()
    print(f"===== {arm.name} =====", flush=True)
    train_arm_wl(table, arm, pool, BASE)
print("all arms saved.")

[no_branches] cached, skip
[shuffled_targets] cached, skip
[zipf_input] cached, skip
[decisive_only] cached, skip
all arms saved.


In [4]:
# §3 — evaluate every saved arm on the within-lane TEST rows (decisive only), per lane.
# lane_const comes from the lane's TRAIN rows — fair here by construction.
import joblib  # noqa: F401  (parity with the source probe's imports)

RRF_DELTA = 0.15


def _serve(probs, thr, delta=RRF_DELTA):
    base = np.asarray(serve_from_probabilities(probs, thr))
    d = probs["dense_only"].to_numpy(); sp = probs["sparse_only"].to_numpy()
    return np.where(np.abs(d - sp) < delta, "pure_rrf", base)


def eval_arm_within(arm_dir, table, train_mask, test_mask):
    p = _P(arm_dir); meta = _json.loads((p / "meta.json").read_text())
    assert meta.get("train_version") == TRAIN_VERSION, f"{p.name}: stale artifact — retrain"
    er = EncoderRouter.load(p / "router.pt")
    svd = NgramSvd.load(p / "svd.joblib"); thr = np.load(p / "thresholds.npy")
    frame = table.frame
    dec = ((frame["shape"] == "routes_differ") & frame["serve"].notna()).to_numpy()
    idx = np.flatnonzero(test_mask & dec)
    emb = QueryEmbeddings(meta["embedding_model"]).matrix(frame)
    q = frame.loc[idx, "query"]
    blocks = [emb[idx], svd.transform(q)]
    if meta.get("zipf_inputs"):
        z = ZipfStats().frame(q).to_numpy(np.float32)
        blocks.append((z - np.load(p / "zipf_mean.npy")) / np.load(p / "zipf_std.npy"))
    served = _serve(er.probabilities(np.concatenate(blocks, axis=1).astype(np.float32)), thr)

    tf = frame.loc[idx].assign(served=served)
    lanes = []
    for lane, g in tf.groupby("dataset"):
        tr = frame[(frame["dataset"] == lane) & train_mask & dec[np.arange(len(frame))]]
        if len(g) < 30 or tr.empty:
            continue
        consts = {r: tr[f"score_{r}"].mean() for r in ROUTES}   # constants from TRAIN rows
        lane_route = max(consts, key=consts.get)
        sc = {r: g[f"score_{r}"].to_numpy() for r in ROUTES}
        lane_const = float(sc[lane_route].mean())               # applied to TEST rows
        cap = float(np.array([sc[r][i] for i, r in enumerate(g["served"])]).mean())
        orc = float(np.column_stack([sc[r] for r in ROUTES]).max(1).mean())
        lanes.append({"lane": lane, "n": len(g), "captured": cap, "lane_const": lane_const,
                      "oracle": orc, "lift": cap - lane_const,
                      "H": (cap - lane_const) / (orc - lane_const)
                           if (orc - lane_const) > 0.02 else np.nan})
    per_lane = pd.DataFrame(lanes)
    pooled = {"arm": p.name, "lanes": len(per_lane), "n": int(per_lane["n"].sum()),
              "captured": float(np.average(per_lane["captured"], weights=per_lane["n"])),
              "lane_const": float(np.average(per_lane["lane_const"], weights=per_lane["n"])),
              "oracle": float(np.average(per_lane["oracle"], weights=per_lane["n"])),
              "lanes_beating_const": int((per_lane["lift"] > 0.005).sum()),
              "mean_H": float(per_lane["H"].mean())}
    return pooled, per_lane


summaries, per_lane_tables = [], {}
for arm, _ in PROBE_ARMS:
    d = BASE / arm.name
    if not (d / "meta.json").exists():
        print(f"[{arm.name}] not saved — run the training cell")
        continue
    pooled, per_lane = eval_arm_within(d, table, train_mask, test_mask)
    summaries.append(pooled); per_lane_tables[arm.name] = per_lane

print("pooled over within-lane TEST decisive rows "
      "(H = headroom capture over the lane's own train-split constant):")
display(pd.DataFrame(summaries).round(3))
print("\nper-lane detail, base arm:")
display(per_lane_tables.get("no_branches", pd.DataFrame()).sort_values("n", ascending=False).round(3))

pooled over within-lane TEST decisive rows (H = headroom capture over the lane's own train-split constant):


,arm,lanes,n,captured,lane_const,oracle,lanes_beating_const,mean_H
0,no_branches,23,11152,0.448,0.485,0.601,2,-0.604
1,shuffled_targets,23,11152,0.416,0.485,0.601,2,-0.822
2,zipf_input,23,11152,0.435,0.485,0.601,2,-0.744
3,decisive_only,23,11152,0.453,0.485,0.601,4,-0.476



per-lane detail, base arm:


,lane,n,captured,lane_const,oracle,lift,H
19,scirgen-geo-en,2277,0.270,0.266,0.403,0.005,0.034
16,quest,1914,0.549,0.546,0.683,0.003,0.019
3,crumb-legal-qa,1218,0.335,0.409,0.464,-0.074,-1.331
6,finder,916,0.310,0.338,0.485,-0.028,-0.188
18,rarb-math,825,0.500,0.487,0.722,0.014,0.058
2,crumb-code-retrieval,733,0.721,0.729,0.758,-0.009,-0.298
9,gooaq,640,0.359,0.535,0.617,-0.176,-2.137
22,webfaq-eng,493,0.440,0.629,0.752,-0.189,-1.541
15,orcas,414,0.505,0.589,0.711,-0.084,-0.692
11,lotte-technology-forum,358,0.506,0.554,0.644,-0.048,-0.528


## 4 · Score-regressor arm — margins as targets, depth-100 as fallback labels

Regression keeps what the classification heads discard: a row scoring dense 0.9 /
sparse 0.1 and one scoring 0.51 / 0.49 are identical `ok_*` targets but very
different regression targets. Trained on the cascade's route-decided rows only
(`l1`, `argmax_l1_l2`, `depth100_fallback` — the tie-ablation lesson applied), with
the d100 rows entering through their own layer's scores.

**Cross-layer scale guard**: depth-100 scores are NDCG@100 units, layer-1 scores are
serving-objective units. Each layer gets ONE affine normalization (layer-wide
mean/std applied to all three route columns together), which equalizes scale across
layers while preserving every row's within-row ordering and relative margins exactly
— the argmax semantics are untouched.

**Serving**: argmax of predicted scores, gated on predicted margin — below the gate,
fall back to the `pure_rrf` hedge. The gate is SWEPT (margins are in normalized
units, so the old 0.4-in-score-units gate has no direct equivalent); read the
capture-vs-gate curve rather than trusting any single value.

**Two-target design**: `score_pure_rrf` is ignored as a target and rrf is
never argmax-eligible — the decision is dense vs sparse only, with rrf as the
low-margin hedge. Forfeits the ~2% merit-rrf wins; matches the symmetric
dense/sparse + rrf-hedge serving policy the router work converged on.

In [14]:
from lightgbm import LGBMRegressor

REG_LAYERS = ("l1", "argmax_l1_l2", "depth100_fallback")
GATES = (0.0, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.4)
SCORE_COLS = [f"score_{r}" for r in ROUTES]
# rrf is HEDGE-ONLY: never a regression target, never argmax-eligible — the
# decision is dense-vs-sparse, and low margin falls back to pure_rrf.
DECISION_ROUTES = ("dense_only", "sparse_only")
T_IDX = [ROUTES.index(r) for r in DECISION_ROUTES]

# per-layer affine normalization: one (mean, std) per layer over ALL route scores
# jointly — cross-layer scale equalized, within-row ordering/margins preserved.
targets = frame[SCORE_COLS].to_numpy(np.float64).copy()
layers = frame["label_layer"].to_numpy()
for L in np.unique(layers):
    m = layers == L
    mu, sd = targets[m].mean(), targets[m].std()
    targets[m] = (targets[m] - mu) / (sd if sd > 0 else 1.0)

reg_pool = train_mask & frame["label_layer"].isin(REG_LAYERS).to_numpy()
print(f"regressor training rows: {reg_pool.sum():,} "
      f"({dict(frame.loc[reg_pool, 'label_layer'].value_counts())})")

emb_r = QueryEmbeddings(PROBE_ARMS[0][0].embedding_model).matrix(frame)
svd_r = NgramSvd(seed=SEED).fit(frame.loc[reg_pool, "query"])
x_r = np.concatenate([emb_r, svd_r.transform(frame["query"])], axis=1).astype(np.float32)
regs = [LGBMRegressor(n_estimators=400, learning_rate=0.05, random_state=SEED,
                      n_jobs=-1, verbose=-1).fit(x_r[reg_pool], targets[reg_pool, i])
        for i in T_IDX]

# predict on the within-lane TEST decisive rows; serve = gated argmax, rrf fallback
dec_r = ((frame["shape"] == "routes_differ") & frame["serve"].notna()).to_numpy()
idx_r = np.flatnonzero(test_mask & dec_r)
pred = np.column_stack([r.predict(x_r[idx_r]) for r in regs])
order = np.argsort(-pred, axis=1)
best_route = np.array(DECISION_ROUTES)[order[:, 0]]
margins = pred[np.arange(len(idx_r)), order[:, 0]] - pred[np.arange(len(idx_r)), order[:, 1]]
print(f"predicted-margin distribution: p25={np.percentile(margins, 25):.3f} "
      f"median={np.median(margins):.3f} p75={np.percentile(margins, 75):.3f}")


def lane_scores(served):
    tf = frame.loc[idx_r].assign(served=served)
    rows = []
    for lane, g in tf.groupby("dataset"):
        tr = frame[(frame["dataset"] == lane) & train_mask & dec_r]
        if len(g) < 30 or tr.empty:
            continue
        consts = {r: tr[f"score_{r}"].mean() for r in ROUTES}
        lane_route = max(consts, key=consts.get)
        sc = {r: g[f"score_{r}"].to_numpy() for r in ROUTES}
        lane_const = float(sc[lane_route].mean())
        cap = float(np.array([sc[r][i] for i, r in enumerate(g["served"])]).mean())
        orc = float(np.column_stack([sc[r] for r in ROUTES]).max(1).mean())
        rows.append({"lane": lane, "n": len(g), "captured": cap, "lane_const": lane_const,
                     "oracle": orc, "lift": cap - lane_const,
                     "H": (cap - lane_const) / (orc - lane_const)
                          if (orc - lane_const) > 0.02 else np.nan})
    return pd.DataFrame(rows)


curve, per_gate = [], {}
for gate in GATES:
    served = np.where(margins >= gate, best_route, "pure_rrf")
    pl = lane_scores(served)
    per_gate[gate] = pl
    curve.append({"gate": gate,
                  "router_share": float((margins >= gate).mean()),
                  "captured": float(np.average(pl["captured"], weights=pl["n"])),
                  "lane_const": float(np.average(pl["lane_const"], weights=pl["n"])),
                  "lanes_beating_const": int((pl["lift"] > 0.005).sum()),
                  "mean_H": float(pl["H"].mean())})
curve = pd.DataFrame(curve)
print("\ncapture vs margin gate (router_share = rows the regressor decides; rest -> rrf hedge):")
display(curve.round(3))

best_gate = curve.loc[curve["mean_H"].idxmax(), "gate"]
print(f"\nbest gate by mean_H: {best_gate} — per-lane detail:")
display(per_gate[best_gate].sort_values("n", ascending=False).round(3))
print("\ncompare against §3: no_branches mean_H -0.60, decisive_only -0.48, "
      "shuffled floor -0.82; positive mean_H here would be the first real router signal.")

regressor training rows: 44,866 ({'l1': np.int64(34347), 'argmax_l1_l2': np.int64(6516), 'depth100_fallback': np.int64(4003)})
predicted-margin distribution: p25=0.233 median=0.499 p75=0.845

capture vs margin gate (router_share = rows the regressor decides; rest -> rrf hedge):


,gate,router_share,captured,lane_const,lanes_beating_const,mean_H
0,0.000,1.000,0.484,0.485,4,-0.074
1,0.005,0.995,0.485,0.485,4,-0.063
2,0.010,0.990,0.485,0.485,5,-0.059
3,0.020,0.979,0.485,0.485,4,-0.046
4,0.050,0.946,0.487,0.485,5,-0.033
5,0.100,0.894,0.488,0.485,6,-0.039
6,0.200,0.784,0.490,0.485,9,0.014
7,0.400,0.589,0.487,0.485,6,0.022



best gate by mean_H: 0.4 — per-lane detail:


,lane,n,captured,lane_const,oracle,lift,H
19,scirgen-geo-en,2277,0.268,0.266,0.403,0.002,0.016
16,quest,1914,0.551,0.546,0.683,0.005,0.034
3,crumb-legal-qa,1218,0.408,0.409,0.464,-0.000,-0.004
6,finder,916,0.343,0.338,0.485,0.006,0.039
18,rarb-math,825,0.514,0.487,0.722,0.027,0.116
2,crumb-code-retrieval,733,0.726,0.729,0.758,-0.004,-0.127
9,gooaq,640,0.521,0.535,0.617,-0.013,-0.162
22,webfaq-eng,493,0.622,0.629,0.752,-0.007,-0.058
15,orcas,414,0.592,0.589,0.711,0.003,0.026
11,lotte-technology-forum,358,0.555,0.554,0.644,0.001,0.015



compare against §3: no_branches mean_H -0.60, decisive_only -0.48, shuffled floor -0.82; positive mean_H here would be the first real router signal.


In [15]:
# §5 — multi-seed error bars on the regressor's within-lane result.
# Five fresh 80/20 splits + refits; the +0.017 mean_H is real only if the
# 5-seed interval clears zero (instrument-validation: single fits wobble ±0.03).
GATE = 0.02
N_SEEDS = 5


def fit_regressor(pool_mask, seed):
    svd = NgramSvd(seed=seed).fit(frame.loc[pool_mask, "query"])
    x = np.concatenate([emb_r, svd.transform(frame["query"])], axis=1).astype(np.float32)
    regs = [LGBMRegressor(n_estimators=400, learning_rate=0.05, random_state=seed,
                          n_jobs=-1, verbose=-1).fit(x[pool_mask], targets[pool_mask, i])
            for i in T_IDX]
    return x, regs


def gated_serve(x, regs, idx, gate):
    pred = np.column_stack([r.predict(x[idx]) for r in regs])
    order = np.argsort(-pred, axis=1)
    best = np.array(DECISION_ROUTES)[order[:, 0]]
    marg = pred[np.arange(len(idx)), order[:, 0]] - pred[np.arange(len(idx)), order[:, 1]]
    return np.where(marg >= gate, best, "pure_rrf")


def lane_scores_at(idx, served, tr_mask):
    tf = frame.loc[idx].assign(served=served)
    rows = []
    for lane, g in tf.groupby("dataset"):
        tr = frame[(frame["dataset"] == lane) & tr_mask & dec_r]
        if len(g) < 30 or tr.empty:
            continue
        consts = {r: tr[f"score_{r}"].mean() for r in ROUTES}
        lane_route = max(consts, key=consts.get)
        sc = {r: g[f"score_{r}"].to_numpy() for r in ROUTES}
        lane_const = float(sc[lane_route].mean())
        cap = float(np.array([sc[r][i] for i, r in enumerate(g["served"])]).mean())
        orc = float(np.column_stack([sc[r] for r in ROUTES]).max(1).mean())
        rows.append({"lane": lane, "n": len(g), "captured": cap, "lane_const": lane_const,
                     "lift": cap - lane_const,
                     "H": (cap - lane_const) / (orc - lane_const)
                          if (orc - lane_const) > 0.02 else np.nan})
    return pd.DataFrame(rows)


seed_rows = []
for seed in range(N_SEEDS):
    srng = np.random.default_rng(seed)
    tmask = np.zeros(len(frame), dtype=bool)
    for _, ix in frame.groupby("dataset").indices.items():
        tmask[srng.choice(ix, size=max(1, int(len(ix) * TEST_FRAC)), replace=False)] = True
    trmask = ~tmask
    pool = trmask & frame["label_layer"].isin(REG_LAYERS).to_numpy()
    x, regs = fit_regressor(pool, seed)
    idx = np.flatnonzero(tmask & dec_r)
    pl = lane_scores_at(idx, gated_serve(x, regs, idx, GATE), trmask)
    seed_rows.append({"seed": seed,
                      "captured": float(np.average(pl["captured"], weights=pl["n"])),
                      "lane_const": float(np.average(pl["lane_const"], weights=pl["n"])),
                      "pooled_lift": float(np.average(pl["lift"], weights=pl["n"])),
                      "lanes_beating_const": int((pl["lift"] > 0.005).sum()),
                      "mean_H": float(pl["H"].mean())})
    print(f"seed {seed}: mean_H={seed_rows[-1]['mean_H']:+.3f}  "
          f"pooled_lift={seed_rows[-1]['pooled_lift']:+.4f}  "
          f"lanes_beating={seed_rows[-1]['lanes_beating_const']}", flush=True)

sr = pd.DataFrame(seed_rows)
mh, sd = sr["mean_H"].mean(), sr["mean_H"].std()
print(f"\nmean_H over {N_SEEDS} seeds at gate {GATE}: {mh:+.3f} ± {sd:.3f} "
      f"({'CLEARS zero — within-lane signal is real' if mh - sd > 0 else 'does NOT clear zero — noise-level'})")
display(sr.round(4))

seed 0: mean_H=-0.046  pooled_lift=+0.0005  lanes_beating=4
seed 1: mean_H=-0.163  pooled_lift=-0.0039  lanes_beating=5
seed 2: mean_H=-0.093  pooled_lift=+0.0038  lanes_beating=6
seed 3: mean_H=-0.124  pooled_lift=-0.0039  lanes_beating=3
seed 4: mean_H=-0.041  pooled_lift=+0.0001  lanes_beating=6

mean_H over 5 seeds at gate 0.02: -0.093 ± 0.052 (does NOT clear zero — noise-level)


,seed,captured,lane_const,pooled_lift,lanes_beating_const,mean_H
0,0,0.4852,0.4847,0.0005,4,-0.0459
1,1,0.4823,0.4863,-0.0039,5,-0.1631
2,2,0.4915,0.4878,0.0038,6,-0.0933
3,3,0.4903,0.4943,-0.0039,3,-0.1244
4,4,0.4814,0.4813,0.0001,6,-0.0405


In [16]:
# §6 — held-out-lane regressor: does the within-lane result survive an unseen collection,
# or was the regressor reconstructing per-lane constants through topic proxies?
# Per holdout: train on every OTHER lane's route-decided rows, evaluate on the held-out
# lane's decisive rows. hr_vs_lane is the oracle bar; lift_vs_global the deployable one.
HOLDOUTS = ("rarb-math", "gooaq", "quest", "finder", "crumb-legal-qa")
GATES_H = (0.0, 0.02, 0.1)

holdout_rows = []
for lane in HOLDOUTS:
    in_lane = (frame["dataset"] == lane).to_numpy()
    pool = ~in_lane & frame["label_layer"].isin(REG_LAYERS).to_numpy()
    x, regs = fit_regressor(pool, SEED)
    idx = np.flatnonzero(in_lane & dec_r)
    tf = frame.loc[idx]
    sc = {r: tf[f"score_{r}"].to_numpy() for r in ROUTES}
    consts = {r: float(sc[r].mean()) for r in ROUTES}
    lane_route = max(consts, key=consts.get)
    lane_const, oracle = consts[lane_route], float(
        np.column_stack([sc[r] for r in ROUTES]).max(1).mean())
    trd = frame[pool]
    global_route = max(ROUTES, key=lambda r: trd[f"score_{r}"].mean())
    for gate in GATES_H:
        served = gated_serve(x, regs, idx, gate)
        cap = float(np.array([sc[r][i] for i, r in enumerate(served)]).mean())
        holdout_rows.append({
            "holdout": lane, "gate": gate, "n_dec": len(idx), "captured": cap,
            "lane_route": lane_route, "lane_const": lane_const,
            "lift_vs_lane": cap - lane_const,
            "global_route": global_route, "global_const": consts[global_route],
            "lift_vs_global": cap - consts[global_route], "oracle": oracle,
            "router_share": float((served != "pure_rrf").mean()),
        })
    print(f"[{lane}] done", flush=True)

ho = pd.DataFrame(holdout_rows)
print("\nheld-out-lane regressor (compare: classification arms were -0.03 to -0.07 "
      "lift_vs_lane on rarb-math):")
display(ho.round(3))
print("\nverdict per gate (mean over holdout lanes):")
display(ho.groupby("gate")[["lift_vs_lane", "lift_vs_global"]].mean().round(4))

[rarb-math] done
[gooaq] done
[quest] done
[finder] done
[crumb-legal-qa] done

held-out-lane regressor (compare: classification arms were -0.03 to -0.07 lift_vs_lane on rarb-math):


,holdout,gate,n_dec,captured,lane_route,lane_const,lift_vs_lane,global_route,global_const,lift_vs_global,oracle,router_share
0,rarb-math,0.00,4156,0.457,pure_rrf,0.492,-0.034,pure_rrf,0.492,-0.034,0.725,1.000
1,rarb-math,0.02,4156,0.458,pure_rrf,0.492,-0.034,pure_rrf,0.492,-0.034,0.725,0.999
2,rarb-math,0.10,4156,0.459,pure_rrf,0.492,-0.033,pure_rrf,0.492,-0.033,0.725,0.990
3,gooaq,0.00,3093,0.530,dense_only,0.538,-0.008,pure_rrf,0.412,0.119,0.634,1.000
4,gooaq,0.02,3093,0.530,dense_only,0.538,-0.008,pure_rrf,0.412,0.118,0.634,0.992
5,gooaq,0.10,3093,0.531,dense_only,0.538,-0.007,pure_rrf,0.412,0.119,0.634,0.962
6,quest,0.00,9669,0.300,sparse_only,0.545,-0.245,dense_only,0.222,0.078,0.683,1.000
7,quest,0.02,9669,0.305,sparse_only,0.545,-0.240,dense_only,0.222,0.083,0.683,0.954
8,quest,0.10,9669,0.338,sparse_only,0.545,-0.206,dense_only,0.222,0.116,0.683,0.754
9,finder,0.00,4569,0.298,dense_only,0.336,-0.039,pure_rrf,0.318,-0.020,0.490,1.000



verdict per gate (mean over holdout lanes):


,lift_vs_lane,lift_vs_global
gate,,
0.00,-0.0898,0.0272
0.02,-0.0874,0.0296
0.10,-0.0774,0.0397


In [17]:
# §7a — EYE TEST, part A: read the regressor's actual decisions on real test rows
# (uses §4's seed-0 fit: x_r, regs, idx_r). regret = oracle score − served score.
GATE_EYE = 0.02

pred_e = np.column_stack([r.predict(x_r[idx_r]) for r in regs])
order_e = np.argsort(-pred_e, axis=1)
best_e = np.array(DECISION_ROUTES)[order_e[:, 0]]
marg_e = pred_e[np.arange(len(idx_r)), order_e[:, 0]] - pred_e[np.arange(len(idx_r)), order_e[:, 1]]
served_e = np.where(marg_e >= GATE_EYE, best_e, "pure_rrf")

eye = frame.loc[idx_r, ["dataset", "query", "label_layer", *SCORE_COLS]].copy()
sc_e = eye[SCORE_COLS].to_numpy()
eye["truth"] = np.array(ROUTES)[np.argmax(sc_e, axis=1)]
eye["served"] = served_e
eye["margin"] = marg_e.round(3)
eye["served_score"] = [sc_e[i, ROUTES.index(s)] for i, s in enumerate(served_e)]
eye["regret"] = (sc_e.max(axis=1) - eye["served_score"]).round(3)
pd.set_option("display.max_colwidth", 110)

print("A1 — CONFIDENT WINS (margin > 0.2, served == truth):")
display(eye[(eye["served"] == eye["truth"]) & (eye["margin"] > 0.2)]
        .sort_values("margin", ascending=False)
        [["dataset", "query", "served", "margin", "served_score"]].head(10))

print("A2 — CONFIDENT MISTAKES (margin > 0.2, regret > 0.3) — the load-bearing rows, read the queries:")
display(eye[(eye["margin"] > 0.2) & (eye["regret"] > 0.3)]
        .sort_values(["regret", "margin"], ascending=False)
        [["dataset", "query", "served", "truth", "margin", "regret"]].head(12))

print("A3 — GATED to rrf (margin < gate): does low confidence look like genuine ambiguity?")
display(eye[eye["margin"] < GATE_EYE].sample(min(8, int((eye['margin'] < GATE_EYE).sum())),
                                             random_state=0)
        [["dataset", "query", "truth", "margin", "regret"]])

print("regret by decision bucket — the gate is doing its job if gated regret >= served regret:")
display(eye.assign(bucket=np.where(eye["margin"] >= GATE_EYE, "served", "gated->rrf"))
        .groupby("bucket")["regret"].agg(["mean", "median", "count"]).round(3))

A1 — CONFIDENT WINS (margin > 0.2, served == truth):


,dataset,query,served,margin,served_score
1746,crumb-code-retrieval,"You are given two strings $s$ and $t$, each of length $n$ and consisting of lowercase Latin alphabets. You...",dense_only,2.135,0.832377
1787,crumb-code-retrieval,"We have N balls. The i-th ball has an integer A_i written on it.\n\nFor each k=1, 2, ..., N, solve the fol...",dense_only,2.108,1.000000
1680,crumb-code-retrieval,"Since Sonya is interested in robotics too, she decided to construct robots that will read and recognize nu...",dense_only,2.009,0.789547
3466,crumb-code-retrieval,"You are given string s. Let's call word any largest sequence of consecutive symbols without symbols ',' (c...",dense_only,1.983,1.000000
2816,crumb-code-retrieval,"You are given a sequence of positive integers of length N, a = (a_1, a_2, ..., a_N).\nYour objective is to...",dense_only,1.979,1.000000
2068,crumb-code-retrieval,Devu has an array A consisting of N positive integers. He would like to perform following operation on arr...,dense_only,1.960,1.000000
77785,rarb-code,Write a function to find the maximum sum in the given right triangle of numbers.,dense_only,1.954,1.000000
2567,crumb-code-retrieval,There is a rectangular grid of size $n \times m$. Each cell has a number written on it; the number on the ...,dense_only,1.933,1.000000
2959,crumb-code-retrieval,You are given an array $a[0 \ldots n-1]$ of length $n$ which consists of non-negative integers. Note that ...,dense_only,1.879,0.979171
2527,crumb-code-retrieval,Polycarp wrote on the board a string $s$ containing only lowercase Latin letters ('a'-'z'). This string is...,dense_only,1.816,1.000000


A2 — CONFIDENT MISTAKES (margin > 0.2, regret > 0.3) — the load-bearing rows, read the queries:


,dataset,query,served,truth,margin,regret
79977,rarb-math,Problem: Find all the solutions to\n\[\arctan \frac{1}{x} + \arctan \frac{1}{x + 2} = \arctan \frac{4}{x +...,dense_only,sparse_only,1.018,1.0
79285,rarb-math,"Problem: If $\frac{3x^2-4x+1}{x-1}=m$, and $x$ can be any real number except $1$, what real values can $m$...",dense_only,sparse_only,0.967,1.0
80287,rarb-math,Problem: What is the sum of all integer values of $x$ such that $\frac{67}{2x - 23}$ is an integer?,dense_only,pure_rrf,0.945,1.0
14600,orcas,make my day,dense_only,sparse_only,0.930,1.0
63001,quest,Flowering plants endemic to China discovered by Western botanists,sparse_only,dense_only,0.866,1.0
52932,limit,Who likes Perseverance?,dense_only,sparse_only,0.834,1.0
53455,limit,Who likes Wrenches?,dense_only,sparse_only,0.811,1.0
52690,limit,Who likes Belts?,dense_only,sparse_only,0.802,1.0
53044,limit,Who likes Markhors?,dense_only,sparse_only,0.800,1.0
53680,limit,Who likes Barbers?,dense_only,sparse_only,0.797,1.0


A3 — GATED to rrf (margin < gate): does low confidence look like genuine ambiguity?


,dataset,query,truth,margin,regret
54923,lotte-technology-forum,Can I redirect output to a log file and background a process at the same time?,dense_only,0.011,0.000
83154,scirgen-geo-en,What is your assessment of the potential for the Maximum Entropy Production (MEP) model to serve as a glob...,sparse_only,0.002,0.022
24084,scirgen-geo-en,How do the characteristics of the land surface and atmospheric conditions enable or hinder the applicabili...,dense_only,0.015,0.022
21245,scirgen-geo-en,Who is responsible for coordinating an international network aimed at advancing alpine hydrology research?,sparse_only,0.013,0.028
67212,quest,Sunbird species found in Australia and Southeast Asia,pure_rrf,0.006,0.000
61844,quest,Shane Dawson essay book not about technology,pure_rrf,0.007,0.000
86598,scirgen-geo-en,What is your evaluation of the integration of IoT techniques in developing smart devices for ecosystem mon...,sparse_only,0.000,0.039
48456,finder,"Digital Realty Trust’s occupancy & rental strategy, ticker DLR, is focused on competitive data centers.",sparse_only,0.001,0.811


regret by decision bucket — the gate is doing its job if gated regret >= served regret:


,mean,median,count
bucket,,,
gated->rrf,0.141,0.028,362
served,0.120,0.000,11103


In [18]:
# §7b — fit_router(): fit a regressor (global or lane-specialized) and get back a
# route() function for YOUR OWN raw query strings. Embeds exactly as training did
# (same prefix, normalize_embeddings=True). Scores are normalized layer units —
# only their ordering and margins mean anything.
from sentence_transformers import SentenceTransformer

_MODEL = PROBE_ARMS[0][0].embedding_model
_PREFIX = EMBEDDING_PREFIXES.get(_MODEL, "")
_ST_EYE = SentenceTransformer(_MODEL)


def fit_router(lane=None, gate=0.02, seed=SEED):
    """lane=None -> global pooled regressor (the §4 setup);
    lane='rarb-math' -> specialized on that lane's train rows only.
    Returns route(queries: list[str]) -> DataFrame with scores, route, margin."""
    pool = train_mask & frame["label_layer"].isin(REG_LAYERS).to_numpy()
    if lane is not None:
        pool &= (frame["dataset"] == lane).to_numpy()
    assert pool.sum() >= 100, f"only {pool.sum()} training rows for lane={lane!r}"
    svd = NgramSvd(seed=seed).fit(frame.loc[pool, "query"])
    x = np.concatenate([emb_r, svd.transform(frame["query"])], axis=1).astype(np.float32)
    models = [LGBMRegressor(n_estimators=400, learning_rate=0.05, random_state=seed,
                            n_jobs=-1, verbose=-1).fit(x[pool], targets[pool, i])
              for i in T_IDX]
    print(f"fitted on {pool.sum():,} rows ({'lane=' + lane if lane else 'ALL lanes pooled'})")

    def route(queries, gate=gate):
        e = _ST_EYE.encode([_PREFIX + q for q in queries],
                           normalize_embeddings=True, show_progress_bar=False)
        X = np.concatenate([np.asarray(e, np.float32),
                            svd.transform(pd.Series(queries))], axis=1).astype(np.float32)
        P = np.column_stack([m.predict(X) for m in models])
        o = np.argsort(-P, axis=1)
        marg = P[np.arange(len(queries)), o[:, 0]] - P[np.arange(len(queries)), o[:, 1]]
        return pd.DataFrame({
            "query": queries,
            **{r: P[:, i].round(3) for i, r in enumerate(DECISION_ROUTES)},
            "route": np.where(marg >= gate, np.array(DECISION_ROUTES)[o[:, 0]], "pure_rrf"),
            "margin": marg.round(3),
        })

    return route


route = fit_router()   # global; or fit_router(lane="rarb-math")

fitted on 44,866 rows (ALL lanes pooled)


In [20]:
display(route([
    "CVE-2021-44228 log4j remote code execution",
    "Who likes Wrenches?",
    "how does photosynthesis differ from cellular respiration",
    "Find all the integer roots of 2x^4 + 4x^3 - 5x^2 + 2x - 3 = 0",
    "why do cats purr",
    "I love doing this. Ohoho", 
    "what is qdrant?"
]))

,query,dense_only,sparse_only,route,margin
0,CVE-2021-44228 log4j remote code execution,0.026,0.203,sparse_only,0.178
1,Who likes Wrenches?,-1.052,1.190,sparse_only,2.243
2,how does photosynthesis differ from cellular respiration,0.124,-0.101,dense_only,0.225
3,Find all the integer roots of 2x^4 + 4x^3 - 5x^2 + 2x - 3 = 0,0.197,-0.331,dense_only,0.529
4,why do cats purr,0.536,0.119,dense_only,0.417
5,I love doing this. Ohoho,0.401,-0.064,dense_only,0.465
6,what is qdrant?,0.773,-0.011,dense_only,0.785


## Reading it

- **`mean_H` and `lanes_beating_const` for `no_branches` vs `shuffled_targets`** is the
  headline. Within-lane there are no cross-lane contradictions: if the real arm still
  can't clear the shuffled floor and the per-lane constants, the input representation
  is the wall (rarity blindness) and no label work changes that. If it clears them,
  labels + inputs are jointly sufficient and the generic-router problem reduces to
  cross-lane label structure — the consistency census becomes the next artifact.
- **`zipf_input` vs `no_branches`**: whether query-intrinsic rarity features buy
  anything once lane structure is out of the picture.
- **`decisive_only` vs `no_branches`**: the tie-hygiene ablation — whether the 24K
  qrels-ceiling tie rows ("both acceptable" targets) dilute the heads.
- Caveat: within-lane evaluation is a DIAGNOSTIC, not a deployability claim — the
  router sees each lane's own queries at train time, which the generic spec forbids.
  This probe exists to locate the wall, not to ship a number.